# 02 Elo-Ratings per Web Scraping holen

Dieses Notebook implementiert BD-07. Es scraped historische Nationalteam-Elo-Daten von `eloratings.net`, beschraenkt sie auf die Teams und Datumsbereiche aus dem StatsBomb-Turnier-Scope und schreibt eine joinbare Bronze-Tabelle.

Outputs:

- `data/raw/elo/elo_raw.parquet`
- `data/bronze/elo_ratings.parquet`
- Cache-Dateien unter `data/cache/elo/`

## Methodischer Ansatz

Die Elo-Daten werden als reproduzierbarer Web-Scraping-Schritt verarbeitet. HTML- und Script-Dateien werden analysiert, die referenzierten TSV-Quelldaten werden geladen und in eine tabellarische Struktur ueberfuehrt.

Rohdaten werden zuerst gesichert; daraus entsteht eine normalisierte Bronze-Tabelle mit historischen Pre-Match-Elo-Werten fuer die relevanten Nationalteams.

In [1]:
import re
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

from project_paths import project_root as get_project_root
from pipeline_config import REQUEST_HEADERS, SOURCE_ELO_BASE_URL as SOURCE_BASE_URL, STATSBOMB_TO_ELO_ALIASES

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 20)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
raw_path = base_path / 'data' / 'raw' / 'elo'
cache_path = base_path / 'data' / 'cache' / 'elo'

matches_path = bronze_path / 'statsbomb_matches.parquet'
raw_output_path = raw_path / 'elo_raw.parquet'
bronze_output_path = bronze_path / 'elo_ratings.parquet'

for path in [raw_path, bronze_path, cache_path]:
    path.mkdir(parents=True, exist_ok=True)


{
    'base_path': str(base_path),
    'matches_path': str(matches_path),
    'raw_output': str(raw_output_path),
    'bronze_output': str(bronze_output_path),
}


{'base_path': '/home/jovyan/work',
 'matches_path': '/home/jovyan/work/data/bronze/statsbomb_matches.parquet',
 'raw_output': '/home/jovyan/work/data/raw/elo/elo_raw.parquet',
 'bronze_output': '/home/jovyan/work/data/bronze/elo_ratings.parquet'}

## Relevante Teams und Datumsbereich

Die Teamliste kommt aus den bereits geladenen StatsBomb-Matches. Damit werden nur Teams verarbeitet, die im Projektscope fuer Joins gebraucht werden.

In [2]:
matches = pd.read_parquet(matches_path)

home_teams = matches['home_team_home_team_name'].dropna()
away_teams = matches['away_team_away_team_name'].dropna()
statsbomb_teams = sorted(set(home_teams).union(set(away_teams)))

match_dates = pd.to_datetime(matches['match_date'])
min_match_date = match_dates.min().normalize()
max_match_date = match_dates.max().normalize()

# Ein Vorlauf stellt sicher, dass der letzte Elo-Wert vor dem ersten Scope-Spiel vorhanden ist.
scrape_start_date = min_match_date - pd.Timedelta(days=370)
scrape_end_date = max_match_date + pd.Timedelta(days=1)

{
    'matches': int(matches['match_id'].nunique()),
    'teams': len(statsbomb_teams),
    'min_match_date': min_match_date.date().isoformat(),
    'max_match_date': max_match_date.date().isoformat(),
    'scrape_start_date': scrape_start_date.date().isoformat(),
    'scrape_end_date': scrape_end_date.date().isoformat(),
}

{'matches': 314,
 'teams': 76,
 'min_match_date': '2018-06-14',
 'max_match_date': '2024-07-15',
 'scrape_start_date': '2017-06-09',
 'scrape_end_date': '2024-07-16'}

## HTML-Struktur analysieren

`eloratings.net` liefert auf Team-URLs dieselbe HTML-Huelle. Die eigentlichen Tabellen werden durch `scripts/ratings.js` angefordert. Dieses Notebook speichert die HTML- und Script-Dateien im Cache und verwendet daraus die sichtbare Datenstruktur: `en.teams.tsv` fuer Teamnamen und `<Teamseite>.tsv` fuer historische Match-/Rating-Zeilen.

In [3]:
from pathlib import Path
def cache_file_for(relative_path: str) -> Path:
    safe_name = relative_path.strip('/').replace('/', '__') or 'index.html'
    return cache_path / safe_name


def fetch_text(relative_path: str, use_cache: bool = True, pause_seconds: float = 0.15) -> str:
    cache_file = cache_file_for(relative_path)
    if use_cache and cache_file.exists():
        return cache_file.read_text(encoding='utf-8')

    url = f"{SOURCE_BASE_URL}/{relative_path.lstrip('/')}"
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    response.encoding = 'utf-8'
    text = response.text

    cache_file.parent.mkdir(parents=True, exist_ok=True)
    temp_file = cache_file.with_suffix(cache_file.suffix + '.tmp')
    temp_file.write_text(text, encoding='utf-8')
    temp_file.replace(cache_file)

    time.sleep(pause_seconds)
    return text


main_html = fetch_text('')
soup = BeautifulSoup(main_html, 'html.parser')
script_sources = [script.get('src') for script in soup.find_all('script') if script.get('src')]

ratings_js_path = next(source for source in script_sources if source.endswith('ratings.js'))
ratings_js = fetch_text(ratings_js_path)
required_tsv_files = sorted(set(re.findall(r"file: '([^']+\\.tsv)'", ratings_js)))

{
    'script_sources': script_sources,
    'ratings_js_path': ratings_js_path,
    'required_tsv_files': required_tsv_files,
}

{'script_sources': ['scripts/dygraph.js',
  'scripts/jquery.js',
  'scripts/slick.core.js',
  'scripts/slick.grid.js',
  'scripts/cldr.js',
  'scripts/event.js',
  'scripts/supplemental.js',
  'scripts/globalize.js',
  'scripts/number.js',
  'scripts/date.js',
  'scripts/ratings.js'],
 'ratings_js_path': 'scripts/ratings.js',
 'required_tsv_files': []}

## Teamnamen auf Elo-Seiten mappen

Die Elo-Daten verwenden Laendercodes und englische Seitennamen. Vier StatsBomb-Bezeichnungen unterscheiden sich von der Elo-Schreibweise; diese werden explizit als Alias dokumentiert.

In [4]:
def normalize_key(value: str) -> str:
    normalized = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'[^a-z0-9]+', ' ', normalized.lower()).strip()


def page_name(value: str) -> str:
    replacements = str.maketrans({
        ' ': '_',
        'à': 'a', 'á': 'a', 'â': 'a', 'ã': 'a', 'ä': 'a', 'å': 'a',
        'ç': 'c',
        'è': 'e', 'é': 'e', 'ê': 'e', 'ë': 'e',
        'ì': 'i', 'í': 'i', 'î': 'i', 'ï': 'i',
        'ò': 'o', 'ó': 'o', 'ô': 'o', 'õ': 'o', 'ö': 'o',
        'ù': 'u', 'ú': 'u', 'û': 'u', 'ü': 'u',
        'ñ': 'n',
    })
    return value.translate(replacements)


def parse_team_dictionary(tsv_text: str) -> pd.DataFrame:
    rows = []
    for line in tsv_text.splitlines():
        fields = line.split('\t')
        if not fields or fields[0].endswith('_loc') or len(fields) < 2:
            continue
        code = fields[0]
        names = [name for name in fields[1:] if name]
        primary_name = names[0]
        rows.append({
            'elo_team_code': code,
            'elo_team_name': primary_name,
            'elo_page': page_name(primary_name),
            'elo_name_variants': names,
        })
    return pd.DataFrame(rows)


team_dictionary = parse_team_dictionary(fetch_text('en.teams.tsv'))


name_to_elo = {}
for row in team_dictionary.to_dict('records'):
    for variant in row['elo_name_variants']:
        name_to_elo[normalize_key(variant)] = row

team_mapping_rows = []
missing_teams = []
for statsbomb_team in statsbomb_teams:
    lookup_name = STATSBOMB_TO_ELO_ALIASES.get(statsbomb_team, statsbomb_team)
    elo_row = name_to_elo.get(normalize_key(lookup_name))
    if elo_row is None:
        missing_teams.append(statsbomb_team)
        continue
    team_mapping_rows.append({
        'team_name': statsbomb_team,
        'elo_lookup_name': lookup_name,
        'elo_team_code': elo_row['elo_team_code'],
        'elo_team_name': elo_row['elo_team_name'],
        'elo_page': elo_row['elo_page'],
    })

if missing_teams:
    raise ValueError(f'Keine Elo-Teamzuordnung gefunden fuer: {missing_teams}')

team_mapping = pd.DataFrame(team_mapping_rows).sort_values('team_name').reset_index(drop=True)
team_mapping.head(20)

,team_name,elo_lookup_name,elo_team_code,elo_team_name,elo_page
0,Albania,Albania,AL,Albania,Albania
1,Algeria,Algeria,DZ,Algeria,Algeria
2,Angola,Angola,AO,Angola,Angola
3,Argentina,Argentina,AR,Argentina,Argentina
4,Australia,Australia,AU,Australia,Australia
5,Austria,Austria,AT,Austria,Austria
6,Belgium,Belgium,BE,Belgium,Belgium
7,Bolivia,Bolivia,BO,Bolivia,Bolivia
8,Brazil,Brazil,BR,Brazil,Brazil
9,Burkina Faso,Burkina Faso,BF,Burkina Faso,Burkina_Faso


## Elo-Teamseiten scrapen

Jede Teamseite besitzt eine TSV-Datei mit historischen Spielen. Die Rating-Spalten sind Ratings nach dem Spiel. Fuer die Projektfrage wird daraus zusaetzlich das Rating vor dem Spiel berechnet, weil Teamstaerke vor dem Spiel benoetigt wird.

In [5]:
RAW_ELO_COLUMNS = [
    'year',
    'month',
    'day',
    'team1_code',
    'team2_code',
    'team1_goals',
    'team2_goals',
    'tournament_code',
    'venue_code',
    'team1_elo_change',
    'team1_elo_after',
    'team2_elo_after',
    'team1_rank_change',
    'team2_rank_change',
    'team1_rank_after',
    'team2_rank_after',
]


def parse_int(value):
    if value is None:
        return pd.NA
    cleaned = str(value).replace('\u2212', '-').replace('â\x88\x92', '-')
    if cleaned in {'', '-'}:
        return pd.NA
    return int(cleaned)


def parse_team_match_tsv(tsv_text: str, source_page: str) -> pd.DataFrame:
    rows = []
    for line_number, line in enumerate(tsv_text.splitlines(), start=1):
        if not line.strip() or line.startswith('<!DOCTYPE html>'):
            continue
        fields = line.split('\t')
        if len(fields) < len(RAW_ELO_COLUMNS):
            continue
        row = dict(zip(RAW_ELO_COLUMNS, fields[:len(RAW_ELO_COLUMNS)]))
        row['source_page'] = source_page
        row['source_line_number'] = line_number
        rows.append(row)

    raw = pd.DataFrame(rows)
    if raw.empty:
        return raw

    numeric_columns = [
        'year', 'month', 'day', 'team1_goals', 'team2_goals', 'team1_elo_change',
        'team1_elo_after', 'team2_elo_after', 'team1_rank_after', 'team2_rank_after',
    ]
    for column in numeric_columns:
        raw[column] = raw[column].map(parse_int).astype('Int64')

    valid_dates = (
        raw['year'].notna()
        & raw['month'].between(1, 12)
        & raw['day'].between(1, 31)
    )
    raw = raw[valid_dates].copy()
    raw['match_date'] = pd.to_datetime(
        {
            'year': raw['year'].astype('int64'),
            'month': raw['month'].astype('int64'),
            'day': raw['day'].astype('int64'),
        },
        errors='coerce',
    )
    raw = raw[raw['match_date'].notna()].copy()
    return raw


raw_frames = []
for page in sorted(team_mapping['elo_page'].unique()):
    page_tsv = fetch_text(f'{page}.tsv')
    raw_frames.append(parse_team_match_tsv(page_tsv, page))

elo_raw = pd.concat(raw_frames, ignore_index=True)
elo_raw = elo_raw.drop_duplicates(
    subset=['match_date', 'team1_code', 'team2_code', 'team1_goals', 'team2_goals', 'tournament_code']
).reset_index(drop=True)

elo_raw = elo_raw[
    (elo_raw['match_date'] >= scrape_start_date)
    & (elo_raw['match_date'] <= scrape_end_date)
].copy()

elo_raw.to_parquet(raw_output_path, index=False)

{
    'raw_rows': len(elo_raw),
    'source_pages': int(team_mapping['elo_page'].nunique()),
    'raw_output': 'data/raw/elo/elo_raw.parquet',
}

{'raw_rows': 4182,
 'source_pages': 76,
 'raw_output': 'data/raw/elo/elo_raw.parquet'}

## Bronze-Tabelle normalisieren

Aus jeder Matchzeile entstehen zwei Team-Zeilen. `elo_rating` ist das berechnete Pre-Match-Rating fuer die jeweilige Mannschaft. Diese Tabelle kann per Teamname und Matchdatum gematcht werden; falls kein exakt gleiches Elo-Spiel existiert, kann ein As-of-Join den letzten vorherigen Wert verwenden.

In [6]:
code_to_name = team_dictionary.set_index('elo_team_code')['elo_team_name'].to_dict()
code_to_statsbomb_team = team_mapping.set_index('elo_team_code')['team_name'].to_dict()
relevant_codes = set(team_mapping['elo_team_code'])

team1_rows = pd.DataFrame({
    'team_name': elo_raw['team1_code'].map(code_to_statsbomb_team),
    'elo_team_name': elo_raw['team1_code'].map(code_to_name),
    'elo_team_code': elo_raw['team1_code'],
    'opponent_elo_team_name': elo_raw['team2_code'].map(code_to_name),
    'opponent_elo_team_code': elo_raw['team2_code'],
    'match_date': elo_raw['match_date'],
    'elo_rating': elo_raw['team1_elo_after'] - elo_raw['team1_elo_change'],
    'elo_rating_after': elo_raw['team1_elo_after'],
    'elo_change': elo_raw['team1_elo_change'],
    'rank_after': elo_raw['team1_rank_after'],
    'source_page': elo_raw['source_page'],
    'source_team_position': 1,
})

team2_rows = pd.DataFrame({
    'team_name': elo_raw['team2_code'].map(code_to_statsbomb_team),
    'elo_team_name': elo_raw['team2_code'].map(code_to_name),
    'elo_team_code': elo_raw['team2_code'],
    'opponent_elo_team_name': elo_raw['team1_code'].map(code_to_name),
    'opponent_elo_team_code': elo_raw['team1_code'],
    'match_date': elo_raw['match_date'],
    'elo_rating': elo_raw['team2_elo_after'] + elo_raw['team1_elo_change'],
    'elo_rating_after': elo_raw['team2_elo_after'],
    'elo_change': -elo_raw['team1_elo_change'],
    'rank_after': elo_raw['team2_rank_after'],
    'source_page': elo_raw['source_page'],
    'source_team_position': 2,
})

elo_ratings = pd.concat([team1_rows, team2_rows], ignore_index=True)
elo_ratings = elo_ratings[elo_ratings['elo_team_code'].isin(relevant_codes)].copy()
elo_ratings['match_date'] = pd.to_datetime(elo_ratings['match_date']).dt.date

elo_ratings = (
    elo_ratings
    .dropna(subset=['team_name', 'elo_rating'])
    .drop_duplicates(subset=['team_name', 'match_date', 'opponent_elo_team_code', 'source_team_position'])
    .sort_values(['team_name', 'match_date', 'opponent_elo_team_name'])
    .reset_index(drop=True)
)

int_columns = ['elo_rating', 'elo_rating_after', 'elo_change', 'rank_after', 'source_team_position']
for column in int_columns:
    elo_ratings[column] = elo_ratings[column].astype('int64')

elo_ratings.to_parquet(bronze_output_path, index=False)

{
    'bronze_rows': len(elo_ratings),
    'teams_with_elo': int(elo_ratings['team_name'].nunique()),
    'bronze_output': 'data/bronze/elo_ratings.parquet',
}

{'bronze_rows': 6182,
 'teams_with_elo': 76,
 'bronze_output': 'data/bronze/elo_ratings.parquet'}

## Qualitaetschecks

Die Checks bilden die BD-07-Akzeptanzkriterien ab: Webquelle vorhanden, Teamnamen und Datumsfelder bereinigt, relevante Teams abgedeckt und per Datum/Team joinbar.

In [7]:
required_columns = ['team_name', 'match_date', 'elo_rating']
missing_required_values = elo_ratings[required_columns].isna().sum().to_dict()
missing_elo_teams = sorted(set(statsbomb_teams) - set(elo_ratings['team_name']))

team_match_dates = pd.concat([
    matches[['match_date', 'home_team_home_team_name']].rename(columns={'home_team_home_team_name': 'team_name'}),
    matches[['match_date', 'away_team_away_team_name']].rename(columns={'away_team_away_team_name': 'team_name'}),
], ignore_index=True)
team_match_dates['match_date'] = pd.to_datetime(team_match_dates['match_date'])

team_match_windows = (
    team_match_dates.groupby('team_name', as_index=False)
    .agg(first_match_date=('match_date', 'min'), last_match_date=('match_date', 'max'))
)
team_elo_windows = (
    elo_ratings.assign(match_date_ts=pd.to_datetime(elo_ratings['match_date']))
    .groupby('team_name', as_index=False)
    .agg(first_elo_date=('match_date_ts', 'min'), last_elo_date=('match_date_ts', 'max'))
)
coverage = team_match_windows.merge(team_elo_windows, on='team_name', how='left')
teams_without_pre_match_elo = coverage[
    coverage['first_elo_date'].isna() | (coverage['first_elo_date'] > coverage['first_match_date'])
]['team_name'].tolist()

quality_checks = {
    'source_base_url': SOURCE_BASE_URL,
    'raw_output_exists': raw_output_path.exists(),
    'bronze_output_exists': bronze_output_path.exists(),
    'required_columns': required_columns,
    'missing_required_values': missing_required_values,
    'statsbomb_teams': len(statsbomb_teams),
    'teams_with_elo': int(elo_ratings['team_name'].nunique()),
    'missing_elo_teams': missing_elo_teams,
    'teams_without_pre_match_elo': teams_without_pre_match_elo,
    'min_elo_date': elo_ratings['match_date'].min().isoformat(),
    'max_elo_date': elo_ratings['match_date'].max().isoformat(),
}

assert quality_checks['raw_output_exists']
assert quality_checks['bronze_output_exists']
assert all(value == 0 for value in missing_required_values.values())
assert not missing_elo_teams
assert not teams_without_pre_match_elo
assert pd.to_datetime(quality_checks['min_elo_date']) <= scrape_start_date

quality_checks

{'source_base_url': 'https://www.eloratings.net',
 'raw_output_exists': True,
 'bronze_output_exists': True,
 'required_columns': ['team_name', 'match_date', 'elo_rating'],
 'missing_required_values': {'team_name': 0, 'match_date': 0, 'elo_rating': 0},
 'statsbomb_teams': 76,
 'teams_with_elo': 76,
 'missing_elo_teams': [],
 'teams_without_pre_match_elo': [],
 'min_elo_date': '2017-06-09',
 'max_elo_date': '2024-07-14'}

In [8]:
elo_ratings.head(20)

,team_name,elo_team_name,elo_team_code,opponent_elo_team_name,opponent_elo_team_code,match_date,elo_rating,elo_rating_after,elo_change,rank_after,source_page,source_team_position
0,Albania,Albania,AL,Israel,IL,2017-06-11,1552,1602,50,60,Albania,2
1,Albania,Albania,AL,Liechtenstein,LI,2017-09-02,1602,1605,3,59,Albania,1
2,Albania,Albania,AL,Macedonia,MK,2017-09-05,1605,1603,-2,61,Albania,2
3,Albania,Albania,AL,Spain,ES,2017-10-06,1603,1600,-3,61,Albania,2
4,Albania,Albania,AL,Italy,IT,2017-10-09,1600,1592,-8,64,Albania,1
5,Albania,Albania,AL,Turkey,TR,2017-11-13,1592,1608,16,60,Albania,2
6,Albania,Albania,AL,Norway,NO,2018-03-26,1608,1595,-13,64,Albania,1
7,Albania,Albania,AL,Kosovo,KO,2018-05-29,1595,1569,-26,68,Albania,2
8,Albania,Albania,AL,Ukraine,UA,2018-06-03,1569,1560,-9,68,Albania,2
9,Albania,Albania,AL,Israel,IL,2018-09-07,1560,1573,13,66,Albania,1


## Ergebnis

BD-07 ist erfuellt, wenn die Assertions erfolgreich sind und `data/raw/elo/elo_raw.parquet` sowie `data/bronze/elo_ratings.parquet` existieren. Die Bronze-Tabelle enthaelt pro relevanter Mannschaft historische Pre-Match-Elo-Werte und ist fuer Joins auf `team_name` und `match_date` vorbereitet.